# Customer Order Insights

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, datediff, current_date, when

spark = SparkSession.builder.appName("CustomerOrderInsightsWeek3").getOrCreate()

In [2]:
from google.colab import files
uploaded = files.upload()
data_dir = "/content"


Saving customers.csv to customers.csv
Saving orders.csv to orders.csv


In [3]:
customers = spark.read.option("header", True).option("inferSchema", True).csv("/content/customers.csv")
orders = spark.read.option("header", True).option("inferSchema", True).csv("/content/orders.csv")

customers.printSchema()
orders.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- expected_delivery_date: date (nullable = true)
 |-- actual_delivery_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- issue_type: string (nullable = true)
 |-- order_amount: integer (nullable = true)



In [4]:
orders = orders.withColumn("order_date", col("order_date").cast("date")) \
               .withColumn("expected_delivery_date", col("expected_delivery_date").cast("date")) \
               .withColumn("actual_delivery_date", col("actual_delivery_date").cast("date"))

order_details = orders.join(customers, on="customer_id", how="left")

order_details = order_details.withColumn(
    "delay_days",
    when(
        col("actual_delivery_date").isNotNull(),
        datediff(col("actual_delivery_date"), col("expected_delivery_date"))
    ).otherwise(
        datediff(current_date(), col("expected_delivery_date"))
    )
).withColumn(
    "is_delayed",
    when(col("delay_days") > 0, True).otherwise(False)
)

order_details.select(
    "order_id", "customer_name", "region", "order_date",
    "expected_delivery_date", "actual_delivery_date", "delay_days", "is_delayed"
).show(10, truncate=False)

+--------+-------------+------+----------+----------------------+--------------------+----------+----------+
|order_id|customer_name|region|order_date|expected_delivery_date|actual_delivery_date|delay_days|is_delayed|
+--------+-------------+------+----------+----------------------+--------------------+----------+----------+
|1001    |Arun Kumar   |South |2026-09-03|2026-09-09            |2026-09-08          |-1        |false     |
|1002    |Meena Ravi   |South |2026-09-05|2026-09-12            |2026-09-11          |-1        |false     |
|1003    |Rahul Sharma |South |2026-09-07|2026-09-12            |NULL                |51        |true      |
|1004    |Priya Nair   |South |2026-09-09|2026-09-15            |2026-09-17          |2         |true      |
|1005    |Karan Shah   |West  |2026-09-11|2026-09-18            |NULL                |45        |true      |
|1006    |Divya Menon  |South |2026-09-13|2026-09-18            |NULL                |45        |true      |
|1007    |Vikram Iy

In [6]:
from pyspark.sql.functions import sum as _sum, avg as _avg, count as _count, col

delays_by_region = order_details.groupBy("region").agg(
    _count("order_id").alias("total_orders"),
    _sum(col("is_delayed").cast("int")).alias("delayed_orders"),
    _avg("delay_days").alias("average_delay_days")
)

delays_by_region.show(truncate=False)

+------+------------+--------------+------------------+
|region|total_orders|delayed_orders|average_delay_days|
+------+------------+--------------+------------------+
|South |18          |13            |21.444444444444443|
|East  |3           |2             |3.0               |
|West  |6           |3             |25.0              |
|North |3           |1             |18.0              |
+------+------------+--------------+------------------+



In [7]:
order_details.write.mode("overwrite").option("header", True).csv("../outputs/joined_order_details")
delays_by_region.write.mode("overwrite").option("header", True).csv("../outputs/delays_by_region")

In [9]:
!zip -r my_output.zip /outputs


  adding: outputs/ (stored 0%)
  adding: outputs/joined_order_details/ (stored 0%)
  adding: outputs/joined_order_details/part-00000-26615e83-afe0-4c29-9071-e459b75beeb6-c000.csv (deflated 71%)
  adding: outputs/joined_order_details/_SUCCESS (stored 0%)
  adding: outputs/joined_order_details/._SUCCESS.crc (stored 0%)
  adding: outputs/joined_order_details/.part-00000-26615e83-afe0-4c29-9071-e459b75beeb6-c000.csv.crc (stored 0%)
  adding: outputs/delays_by_region/ (stored 0%)
  adding: outputs/delays_by_region/_SUCCESS (stored 0%)
  adding: outputs/delays_by_region/part-00000-5e6088f0-111f-4d7c-b533-82043f9796b5-c000.csv (deflated 24%)
  adding: outputs/delays_by_region/._SUCCESS.crc (stored 0%)
  adding: outputs/delays_by_region/.part-00000-5e6088f0-111f-4d7c-b533-82043f9796b5-c000.csv.crc (stored 0%)


In [10]:
from google.colab import files
files.download('my_output.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
spark.stop()